# Step 1.2 — Enriched Feature Matrix
`**1.2a: Structured macro/financial data**`

**Thesis: Geopolitical Turning Points and Macroeconomic Volatility — Extension of Saadaoui (2026)**

Building the high-dimensional feature matrix used in Steps 2–5

**Sample:** 1990-01 to 2022-01 (385 monthly observations, matching the Saadaoui .dta dataset)

---

## Files loaded and output by this notebook can be found in: 
```
project/
├── data/
│   ├── Saadaoui_2026_JCE.dta
│   └── 02_features/
│       └── raw/          ← placing all manual downloads here
│           ├── bdi.csv   (Investing.com: ; delimited, mm/dd/yyyy, comma decimal)
│           └── gscpi.csv (NY Fed: ; delimited, French dates, comma decimal)
└── notebooks/
    └── step1_feature_matrix.ipynb   ← this file
```

## Data sources

| Variable | Source | Format |
|---|---|---|
| Core variables (PRI, WTI, controls) | Saadaoui (2026) `.dta` | 49 variables, already transformed |
| VIX | Yahoo Finance `^VIX` | Daily → monthly mean |
| Gold price (USD/oz) | Datahub.io LBMA monthly | Monthly, auto-download |
| Brent crude | Yahoo Finance `BZ=F` → FRED `DCOILBRENTEU` fallback | Daily → monthly mean |
| T-bill 3m, 10y Treasury, TED, REER, INDPRO | FRED direct CSV | Monthly, auto-download |
| Baltic Dry Index | Investing.com manual CSV | `;` delimited, `mm/dd/yyyy`, comma decimal |
| GSCPI | NY Fed manual CSV | `;` delimited, French dates, comma decimal |
| Global EPU | Baker-Bloom-Davis Excel | Auto-download |

## Derived variables
- `term_spread` = `gs10` − `tb3ms`
- `lreer`, `lgold`, `lbdi`, `lip`, `lepu` = log transforms
- `dvix` = first difference of VIX
- `lbrent` = log Brent crude

---
## Setup

In [31]:
import pandas as pd
import numpy as np
import os
import json
import requests
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pyreadstat

# Paths
RAW   = '../data/02_features/raw'   # manual downloads + auto-cached files
PROC  = '../data/02_features'       # processed outputs
DTA   = '../data/Saadaoui_2026_JCE.dta'

os.makedirs(RAW,  exist_ok=True)
os.makedirs(PROC, exist_ok=True)    

# The Sample window (buffer before 1990-01 allows lags)
START = '1989-01-01'
END   = '2022-03-01'

print('Setup complete.')
print(f'RAW  → {os.path.abspath(RAW)}')
print(f'PROC → {os.path.abspath(PROC)}')

Setup complete.
RAW  → c:\Users\HP\Desktop\replication+contribution\data\02_features\raw
PROC → c:\Users\HP\Desktop\replication+contribution\data\02_features


---
## Section A — Load & Download Raw Data

Each cell skips the download if its output file already exists. Safe to re-run from scratch.

### A1. Saadaoui core dataset
**Source:** Saadaoui (2026), *Journal of Comparative Economics*  
**File:** `../data/Saadaoui_2026_JCE.dta`

In [32]:
if not os.path.exists(DTA):
    raise FileNotFoundError(f'Core dataset not found: {DTA}')

df_core, _ = pyreadstat.read_dta(DTA)

# Stata monthly date: months elapsed since 1960-01-01
df_core['date'] = (pd.to_datetime('1960-01-01')
                   + pd.to_timedelta(df_core['Period'] * 30.4375, unit='D'))
df_core['date'] = df_core['date'].dt.to_period('M').dt.to_timestamp()
df_core = df_core.set_index('date').sort_index()

print(f'Core dataset: {df_core.shape[0]} obs × {df_core.shape[1]} variables')
print(f'Range: {df_core.index[0].date()} → {df_core.index[-1].date()}')
df_core.head(3)

Core dataset: 386 obs × 48 variables
Range: 1989-12-01 → 2022-01-01


,lwip,lgop,lwti,lpri,pri,u_wip,u_gop,pri_jp,pri_aus,pri_cds,...,dlpri_vn,lpri_uk,dlpri_uk,dlpri,llwip,llgop,l2lwip,l2lgop,d2pri,d2pri_jp
date,,,,,,,,,,,,,,,,,,,,,
1989-12-01,4.135466,4.109584,2.876816,-0.568825,-0.6,2.890271,-0.340788,3.4,1.7,-6.3,...,0.0,1.131402,0.0,0.000000,4.139441,4.118796,4.131745,4.124588,-0.1,0.0
1990-01-01,4.139524,4.113802,2.849079,-0.732668,-0.8,0.997853,1.766212,3.4,1.7,-6.3,...,0.0,1.131402,0.0,-0.163843,4.135466,4.109584,4.139441,4.118796,-0.2,0.0
1990-03-01,4.143738,4.128463,2.764880,-0.808867,-0.9,-6.853467,1.590641,3.4,1.7,-6.3,...,0.0,1.131402,0.0,-0.076199,4.139524,4.113802,4.135466,4.109584,0.1,0.0


### A2. VIX — CBOE Volatility Index
**Source:** Yahoo Finance `^VIX` via `yfinance`  
**Format:** Daily CSV with 3 metadata header rows (yfinance v0.2+), then date + Close  
**Frequency:** Daily → resampled to monthly mean in Section B

In [33]:
P_VIX = f'{RAW}/vix.csv'

if not os.path.exists(P_VIX):
    vix = yf.download('^VIX', start=START, end=END, progress=False)
    # yfinance ≥0.2 uses multi-level columns; flatten to single level
    if isinstance(vix.columns, pd.MultiIndex):
        vix.columns = [c[0] for c in vix.columns]
    vix[['Close']].rename(columns={'Close': 'vix'}).to_csv(P_VIX)
    print(f'Downloaded VIX: {len(vix)} daily rows')
else:
    print('ℹVIX: file exists, skipping download.')

# Robust load: skip metadata rows, force numeric, force DatetimeIndex
vix_raw = pd.read_csv(P_VIX, index_col=0, skiprows=lambda i: i in [1, 2])
vix_raw.index = pd.to_datetime(vix_raw.index, errors='coerce')
vix_raw = vix_raw[vix_raw.index.notna()]
price_col = [c for c in vix_raw.columns if c.lower() in ('close', 'vix', 'adj close')][0]
vix_raw[price_col] = pd.to_numeric(vix_raw[price_col], errors='coerce')
vix_raw = vix_raw.dropna(subset=[price_col])

print(f'Loaded VIX: {len(vix_raw)} rows | {vix_raw.index[0].date()} → {vix_raw.index[-1].date()}')
print(f'Range: {vix_raw[price_col].min():.2f} – {vix_raw[price_col].max():.2f}')

ℹVIX: file exists, skipping download.
Loaded VIX: 8101 rows | 1990-01-04 → 2022-02-28
Range: 9.14 – 82.69


### A3. Gold Price (USD/oz)
**Source:** Datahub.io — LBMA monthly fixings  
**URL:** `https://datahub.io/core/gold-prices/_r/-/data/monthly-processed.csv`  
**Format:** Standard CSV, `Date` + `Price` columns, already monthly

In [34]:
P_GOLD = f'{RAW}/gold_monthly.csv'

if not os.path.exists(P_GOLD):
    url = 'https://datahub.io/core/gold-prices/_r/-/data/monthly-processed.csv'
    gold_df = pd.read_csv(url, parse_dates=['Date'])
    gold_df.columns = ['date', 'gold_price']
    gold_df.set_index('date').sort_index().to_csv(P_GOLD)
    print(f'✅ Downloaded gold: {len(gold_df)} rows')
else:
    print('ℹGold: file exists, skipping download.')

gold_raw = pd.read_csv(P_GOLD, index_col=0, parse_dates=True)
gold_raw['gold_price'] = pd.to_numeric(gold_raw['gold_price'], errors='coerce')
gold_raw = gold_raw.dropna()
print(f'Loaded gold: {len(gold_raw)} rows | {gold_raw.index[0].date()} → {gold_raw.index[-1].date()}')
gold_raw.tail(3)

ℹGold: file exists, skipping download.
Loaded gold: 2319 rows | 1833-01-01 → 2026-03-01


,gold_price
date,
2026-01-01,4752.75
2026-02-01,5019.97
2026-03-01,4855.54


### A4. Brent Crude Price
**Source:** FRED DCOILBRENTEU (Europe Brent Spot Price FOB)
**URL:** https://fred.stlouisfed.org/series/DCOILBRENTEU
**Why:** Adds WTI–Brent spread as geopolitical oil market segmentation proxy
**Coverage:** 1987-05-20 to present (daily) → resampled to monthly

In [4]:
P_BRENT = f'{RAW}/brent_fred.csv'

if not os.path.exists(P_BRENT):
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DCOILBRENTEU"
    try:
        brent_fred = pd.read_csv(url, index_col=0, parse_dates=True, na_values=['.', 'NA', ''])
        # Clean: remove duplicate dates, coerce numeric
        brent_fred = brent_fred[~brent_fred.index.duplicated(keep='first')]
        brent_fred['DCOILBRENTEU'] = pd.to_numeric(brent_fred['DCOILBRENTEU'], errors='coerce')
        brent_fred = brent_fred.dropna(subset=['DCOILBRENTEU'])
        brent_fred[['DCOILBRENTEU']].to_csv(P_BRENT)
        print(f"✅ Downloaded Brent from FRED: {len(brent_fred)} daily rows")
    except Exception as e:
        print(f"⚠️ FRED Brent download failed: {e}")
        print("   Keeping existing file if available, or Brent will be excluded.")
else:
    print('ℹ Brent (FRED): file exists, skipping download.')

# Load and process to monthly log series
if os.path.exists(P_BRENT):
    brent_raw = pd.read_csv(P_BRENT, index_col=0, parse_dates=True, na_values=['.', 'NA', ''])
    brent_raw = brent_raw[~brent_raw.index.duplicated(keep='first')]
    brent_raw['DCOILBRENTEU'] = pd.to_numeric(brent_raw['DCOILBRENTEU'], errors='coerce')
    brent_raw = brent_raw.dropna(subset=['DCOILBRENTEU'])
    
    # Resample: daily → monthly mean, then log transform
    brent_monthly = brent_raw['DCOILBRENTEU'].resample('MS').mean()
    monthly_series['lbrent'] = np.log(brent_monthly.replace(0, np.nan)).rename('lbrent')
    
    print(f"lbrent: {monthly_series['lbrent'].notna().sum()} months | "
          f"{monthly_series['lbrent'].first_valid_index().date()} → "
          f"{monthly_series['lbrent'].last_valid_index().date()}")

NameError: name 'RAW' is not defined

### A5. FRED Series — Treasury Yields, TED Spread, REER, Industrial Production
**Source:** FRED direct CSV — no API key required  
**URL pattern:** `https://fred.stlouisfed.org/graph/fredgraph.csv?id=<ID>`

| FRED ID | Variable | Frequency |
|---|---|---|
| `TB3MS` | 3-month T-bill yield (%) | Monthly |
| `GS10` | 10-year Treasury yield (%) | Monthly |
| `TEDRATE` | TED spread (%) | Daily |
| `RBUSBIS` | US REER, BIS broad index | Monthly (starts 1994) |
| `INDPRO` | US Industrial Production Index | Monthly |

In [36]:
FRED_SERIES = {
    'tb3ms'  : f'{RAW}/tb3ms.csv',
    'gs10'   : f'{RAW}/gs10.csv',
    'tedrate': f'{RAW}/tedrate.csv',
    'reer'   : f'{RAW}/reer_bis.csv',
    'indpro' : f'{RAW}/indpro.csv',
}
FRED_URLS = {
    'tb3ms'  : 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=TB3MS',
    'gs10'   : 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=GS10',
    'tedrate': 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=TEDRATE',
    'reer'   : 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=RBUSBIS',
    'indpro' : 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=INDPRO',
}

for name, path in FRED_SERIES.items():
    if not os.path.exists(path):
        r = requests.get(FRED_URLS[name], timeout=15)
        if r.status_code == 200:
            with open(path, 'w') as fh:
                fh.write(r.text)
            n = len(r.text.strip().split('\n')) - 1
            print(f'{name:10s}: downloaded ({n} rows)')
        else:
            print(f'{name:10s}: FAILED — HTTP {r.status_code}')
    else:
        tmp = pd.read_csv(path, na_values=['.', 'NA', ''])
        print(f'ℹ{name:10s}: exists ({len(tmp)} rows)')

ℹtb3ms     : exists (1107 rows)
ℹgs10      : exists (876 rows)
ℹtedrate   : exists (9407 rows)
ℹreer      : exists (386 rows)
ℹindpro    : exists (1287 rows)


### A6. Baltic Dry Index (BDI)
**Source:** Investing.com — manual download  
**Place file at:** `../data/02_features/raw/bdi.csv`  
**URL:** https://www.investing.com/a88cc4bb-e670-4930-b94c-7f13777f497f  
**Format:**
- Delimiter: `;`
- Date column: `mm/dd/yyyy`  
- Value column: comma as **decimal** separator (e.g. `2568,3` = 2568.3)
- Header row: `Date;Value`

In [37]:
P_BDI     = f'{RAW}/bdi.csv'
P_BDI_OUT = f'{RAW}/bdi_clean.csv'

if os.path.exists(P_BDI):
    # Read semicolon-delimited file, treat comma as decimal separator
    bdi_df = pd.read_csv(
        P_BDI,
        sep=';',
        decimal=',',       # comma is decimal separator (e.g. 2568,3 → 2568.3)
        header=0,
        usecols=[0, 1],
        names=['date', 'bdi'],
        skiprows=0,
    )

    # Drop any non-data rows (header repetitions, empty rows)
    bdi_df['date'] = pd.to_datetime(bdi_df['date'], format='%m/%d/%Y', errors='coerce')
    bdi_df['bdi']  = pd.to_numeric(bdi_df['bdi'],  errors='coerce')
    bdi_df = bdi_df.dropna().set_index('date').sort_index()

    if len(bdi_df) > 0:
        bdi_df.to_csv(P_BDI_OUT)
        print(f'✅ BDI: {len(bdi_df)} rows | {bdi_df.index[0].date()} → {bdi_df.index[-1].date()}')
        print(f'   Value range: {bdi_df["bdi"].min():.0f} – {bdi_df["bdi"].max():.0f}')
        print(bdi_df.head(3))
    else:
        print('BDI: parsed 0 valid rows. Check delimiter and decimal format.')
else:
    print('BDI file not found.')
    print(f'   Download from: https://www.investing.com/a88cc4bb-e670-4930-b94c-7f13777f497f')
    print(f'   Save as: {P_BDI}')

✅ BDI: 9479 rows | 1986-12-31 → 2025-05-02
   Value range: 1895 – 33154
                  bdi
date                 
1986-12-31  2568.3000
1987-01-02  2540.1001
1987-01-05  2552.3999


### A7. Global Supply Chain Pressure Index (GSCPI)
**Source:** Federal Reserve Bank of New York  
**Place file at:** `../data/02_features/raw/gscpi.csv`  
**URL:** https://www.newyorkfed.org/research/policy/gscpi  
**Format:**
- Delimiter: `;`
- Date column: French month abbreviations, mixed 2/4-digit years (e.g. `31-janv-98`, `28-Feb-1998`)
- Value column: comma as **decimal** separator (e.g. `-1,09` = −1.09)
- Header row: `Date;GSCPI` (or `Date;Index`)
- Coverage: starts 1998-01

In [38]:
P_GSCPI = f'{RAW}/gscpi.csv'
gscpi_series = None

if os.path.exists(P_GSCPI):
    gscpi_df = pd.read_csv(P_GSCPI, sep=';', header=0, names=['date', 'gscpi'],
                            usecols=[0, 1], skiprows=1)   # skip the header row

    # ── Date parsing ──────────────────────────────────────────────────────────
    # French abbreviations → English; handles both 2-digit and 4-digit years
    FR_TO_EN = {
        'janv': 'Jan', 'fév': 'Feb', 'févr': 'Feb',
        'mars': 'Mar', 'avr': 'Apr', 'mai': 'May',
        'juin': 'Jun', 'juil': 'Jul', 'août': 'Aug', 'aout': 'Aug',
        'sept': 'Sep', 'oct': 'Oct', 'nov': 'Nov',
        'déc': 'Dec', 'dec': 'Dec',
    }

    def parse_gscpi_date(s):
        s = str(s).strip().lower()
        for fr, en in FR_TO_EN.items():
            s = s.replace(fr, en)
        # Normalise capitalisation for pd.to_datetime
        s = s.title()  # '31-Jan-98' or '28-Feb-1998'
        dt = pd.to_datetime(s, dayfirst=True, errors='coerce')
        # Fix 2-digit years: pandas maps 00–68 → 2000–2068, 69–99 → 1969–1999
        # GSCPI starts 1998, so any year ≥ 2069 is wrong — shift back 100 years
        if pd.notna(dt) and dt.year > 2030:
            dt = dt.replace(year=dt.year - 100)
        return dt

    gscpi_df['date']  = gscpi_df['date'].apply(parse_gscpi_date)
    # Comma decimal → float
    gscpi_df['gscpi'] = (gscpi_df['gscpi'].astype(str)
                          .str.replace(',', '.', regex=False)
                          .pipe(pd.to_numeric, errors='coerce'))

    gscpi_df = gscpi_df.dropna().set_index('date').sort_index()
    gscpi_series = gscpi_df['gscpi'].resample('MS').last()

    print(f'GSCPI: {gscpi_series.notna().sum()} months '
          f'| {gscpi_series.first_valid_index().date()} → {gscpi_series.last_valid_index().date()}')
    print(gscpi_df.head(3))
else:
    print('GSCPI file not found — will be excluded.')
    print(f'   Download from: https://www.newyorkfed.org/research/policy/gscpi')
    print(f'   Save as: {P_GSCPI}')

GSCPI: 338 months | 1998-02-01 → 2026-03-01
            gscpi
date             
1998-02-28  -0.44
1998-03-31  -0.06
1998-04-30  -0.12


### A8. Global Economic Policy Uncertainty Index (EPU)
**Source:** Baker, Bloom & Davis — policyuncertainty.com  
**URL:** `https://www.policyuncertainty.com/media/Global_Policy_Uncertainty_Data.xlsx`  
**Why:** Captures policy-specific uncertainty (distinct from financial volatility measured by VIX)  
**Coverage:** Monthly from 1997-01

In [39]:
P_EPU = f'{RAW}/epu_global.xlsx'

if not os.path.exists(P_EPU):
    url = 'https://www.policyuncertainty.com/media/Global_Policy_Uncertainty_Data.xlsx'
    r = requests.get(url, timeout=20)
    if r.status_code == 200:
        with open(P_EPU, 'wb') as fh:
            fh.write(r.content)
        print(f'Downloaded EPU: {len(r.content)/1024:.0f} KB')
    else:
        print(f'EPU download failed: HTTP {r.status_code}')
        print('   Manual: https://www.policyuncertainty.com/global_monthly.html')
        print(f'   Save as: {P_EPU}')
else:
    print('ℹEPU: file exists, skipping download.')

if os.path.exists(P_EPU):
    epu_raw = pd.read_excel(P_EPU)
    print(f'EPU loaded: {len(epu_raw)} rows | columns: {epu_raw.columns.tolist()}')
    print(epu_raw.head(3))

ℹEPU: file exists, skipping download.
EPU loaded: 349 rows | columns: ['Year', 'Month', 'GEPU_current', 'GEPU_ppp']
   Year  Month  GEPU_current   GEPU_ppp
0  1997    1.0     76.059427  79.856772
1  1997    2.0     76.286023  75.828220
2  1997    3.0     66.810541  64.850547


---
## Section B — Merge & Build Feature Matrix

### B1. Helper functions

In [40]:
def ensure_datetimeindex(s):
    """Ensure a Series/DataFrame has a proper DatetimeIndex."""
    s = s.copy()
    if not isinstance(s.index, pd.DatetimeIndex):
        s.index = pd.to_datetime(s.index, errors='coerce')
        s = s[s.index.notna()]
    return s

def to_monthly_mean(series_or_df, col=None):
    """Resample a daily series to monthly mean. Accepts Series or single-column DataFrame."""
    s = series_or_df[col] if col else series_or_df
    if isinstance(s, pd.DataFrame):
        s = s.iloc[:, 0]
    s = ensure_datetimeindex(s.dropna())
    s = pd.to_numeric(s, errors='coerce').dropna()
    return s.resample('MS').mean()

def to_monthly_last(series_or_df, col=None):
    """Resample a daily/monthly series to month-start last value."""
    s = series_or_df[col] if col else series_or_df
    if isinstance(s, pd.DataFrame):
        s = s.iloc[:, 0]
    s = ensure_datetimeindex(s.dropna())
    s = pd.to_numeric(s, errors='coerce').dropna()
    return s.resample('MS').last()

def load_fred(path, col_name):
    """Load a standard FRED CSV and return a monthly Series."""
    df = pd.read_csv(path, index_col=0, parse_dates=True, na_values=['.', 'NA', ''])
    df = ensure_datetimeindex(df)
    df.columns = [col_name]
    df[col_name] = pd.to_numeric(df[col_name], errors='coerce')
    return df[col_name].dropna().resample('MS').last()

print('Helpers defined.')

Helpers defined.


### B2. Build all monthly series

In [41]:
monthly_series = {}

# ── VIX ───────────────────────────────────────────────────────────────────────
if os.path.exists(f'{RAW}/vix.csv'):
    vix_d = pd.read_csv(f'{RAW}/vix.csv', index_col=0, skiprows=lambda i: i in [1, 2])
    vix_d = ensure_datetimeindex(vix_d)
    pcol  = [c for c in vix_d.columns if c.lower() in ('close', 'vix', 'adj close')][0]
    monthly_series['vix'] = to_monthly_mean(vix_d[pcol]).rename('vix')
    print(f"vix        : {monthly_series['vix'].notna().sum()} months")

# ── Gold ──────────────────────────────────────────────────────────────────────
if os.path.exists(f'{RAW}/gold_monthly.csv'):
    gold_d = pd.read_csv(f'{RAW}/gold_monthly.csv', index_col=0, parse_dates=True)
    gold_d = ensure_datetimeindex(gold_d)
    gold_s = to_monthly_last(gold_d['gold_price'])
    monthly_series['lgold'] = np.log(gold_s.replace(0, np.nan)).rename('lgold')
    print(f"lgold      : {monthly_series['lgold'].notna().sum()} months")

# ── Brent ─────────────────────────────────────────────────────────────────────
if os.path.exists(f'{RAW}/brent.csv'):
    brent_d = pd.read_csv(f'{RAW}/brent.csv', index_col=0,
                           skiprows=lambda i: i in [1, 2],
                           na_values=['.', 'NA', ''])
    brent_d = ensure_datetimeindex(brent_d)
    bcol    = [c for c in brent_d.columns
               if c.lower() in ('close', 'brent', 'dcoilbrenteu', 'price')][0]
    brent_m = to_monthly_mean(brent_d[bcol])
    monthly_series['lbrent'] = np.log(brent_m.replace(0, np.nan)).rename('lbrent')
    print(f"lbrent     : {monthly_series['lbrent'].notna().sum()} months")

# ── FRED series ───────────────────────────────────────────────────────────────
fred_load = {
    'tb3ms'  : f'{RAW}/tb3ms.csv',
    'gs10'   : f'{RAW}/gs10.csv',
    'tedrate': f'{RAW}/tedrate.csv',
    'reer'   : f'{RAW}/reer_bis.csv',
    'indpro' : f'{RAW}/indpro.csv',
}
for col, path in fred_load.items():
    if os.path.exists(path):
        monthly_series[col] = load_fred(path, col)
        print(f"{col:10s}: {monthly_series[col].notna().sum()} months")

# ── BDI ───────────────────────────────────────────────────────────────────────
if os.path.exists(f'{RAW}/bdi_clean.csv'):
    bdi_d = pd.read_csv(f'{RAW}/bdi_clean.csv', index_col=0, parse_dates=True)
    bdi_d = ensure_datetimeindex(bdi_d)
    monthly_series['bdi'] = to_monthly_mean(bdi_d['bdi']).rename('bdi')
    print(f"bdi        : {monthly_series['bdi'].notna().sum()} months")

# ── GSCPI ─────────────────────────────────────────────────────────────────────
if gscpi_series is not None and gscpi_series.notna().any():
    monthly_series['gscpi'] = gscpi_series
    print(f"gscpi      : {monthly_series['gscpi'].notna().sum()} months")

# ── EPU ───────────────────────────────────────────────────────────────────────
if os.path.exists(f'{RAW}/epu_global.xlsx'):
    # Read the file without assuming the header row index
    epu_df = pd.read_excel(f'{RAW}/epu_global.xlsx', header=None)

    # Find the row where the first column contains "Year" (the true header)
    header_row = epu_df[epu_df[0] == 'Year'].index[0]

    # Re-read the file, skipping all rows before the header
    epu_df = pd.read_excel(f'{RAW}/epu_global.xlsx', skiprows=header_row)

    # Rename columns (assuming they are in order: Year, Month, GEPU_current, GEPU_ppp)
    epu_df.columns = ['Year', 'Month', 'GEPU_current', 'GEPU_ppp']

    # Convert Year and Month to numeric (coerce errors to NaN)
    epu_df['Year'] = pd.to_numeric(epu_df['Year'], errors='coerce')
    epu_df['Month'] = pd.to_numeric(epu_df['Month'], errors='coerce')

    # Drop rows where Year or Month are missing (including the citation row)
    epu_df = epu_df.dropna(subset=['Year', 'Month'])

    # Create a proper datetime column (first day of the month)
    epu_df['date'] = pd.to_datetime(epu_df[['Year', 'Month']].assign(Day=1))

    # Choose which EPU series to use (PPP‑adjusted is often preferred)
    epu_col = 'GEPU_ppp' if 'GEPU_ppp' in epu_df.columns else 'GEPU_current'

    # Set index and resample to month‑start (already monthly, but ensures alignment)
    epu_s = epu_df.set_index('date')[epu_col].sort_index()
    epu_s = ensure_datetimeindex(epu_s)
    monthly_series['epu'] = epu_s.resample('MS').last().rename('epu')

    print(f"epu        : {monthly_series['epu'].notna().sum()} months")

print(f'\nTotal series collected: {len(monthly_series)}')

vix        : 386 months
lgold      : 2319 months
lbrent     : 175 months
tb3ms     : 1107 months
gs10      : 876 months
tedrate   : 433 months
reer      : 386 months
indpro    : 1287 months
bdi        : 462 months
gscpi      : 338 months
epu        : 347 months

Total series collected: 11


### B3. Derived variables

In [42]:
derived = []

if 'gs10' in monthly_series and 'tb3ms' in monthly_series:
    monthly_series['term_spread'] = (monthly_series['gs10'] - monthly_series['tb3ms']).rename('term_spread')
    derived.append('term_spread')

if 'reer' in monthly_series:
    monthly_series['lreer'] = np.log(monthly_series['reer'].replace(0, np.nan)).rename('lreer')
    derived.append('lreer')

if 'vix' in monthly_series:
    monthly_series['dvix'] = monthly_series['vix'].diff().rename('dvix')
    derived.append('dvix')

if 'bdi' in monthly_series:
    monthly_series['lbdi'] = np.log(monthly_series['bdi'].replace(0, np.nan)).rename('lbdi')
    derived.append('lbdi')

if 'indpro' in monthly_series:
    monthly_series['lip'] = np.log(monthly_series['indpro'].replace(0, np.nan)).rename('lip')
    derived.append('lip')

if 'epu' in monthly_series:
    monthly_series['lepu'] = np.log(monthly_series['epu'].replace(0, np.nan)).rename('lepu')
    derived.append('lepu')

print(f'Derived variables: {derived}')

Derived variables: ['term_spread', 'lreer', 'dvix', 'lbdi', 'lip', 'lepu']


### B4. Merge with core dataset

> **Look-ahead bias note:** All newly added macro controls are lagged 1 month (`shift(1)`) before merging, so that at time *t* the model only sees controls known at *t−1*. The Saadaoui baseline controls (`llwip`, `llgop`, etc.) are already lagged in the original `.dta` file — do **not** lag them again.

In [43]:
# Align core index to month-start timestamps
df_core.index = pd.to_datetime(df_core.index).to_period('M').to_timestamp()
df_enriched = df_core.copy()

for col_name, series in monthly_series.items():
    s = series.copy()
    s = ensure_datetimeindex(s)
    s.index = s.index.to_period('M').to_timestamp()
    s.name = col_name
    df_enriched = df_enriched.join(s, how='left')

# Lag all new macro controls by 1 month to prevent look-ahead bias
new_ctrl_cols = [c for c in monthly_series.keys() if c in df_enriched.columns]
df_enriched[new_ctrl_cols] = df_enriched[new_ctrl_cols].shift(1)

# Restrict to Saadaoui sample
df_enriched = df_enriched.loc['1990-01-01':'2022-01-01']

print(f'Enriched dataset: {df_enriched.shape[0]} obs × {df_enriched.shape[1]} variables')
print(f'Range: {df_enriched.index[0].date()} → {df_enriched.index[-1].date()}')

Enriched dataset: 385 obs × 65 variables
Range: 1990-01-01 → 2022-01-01


### B5. Coverage audit

In [44]:
new_vars = [c for c in monthly_series.keys() if c in df_enriched.columns]

audit = pd.DataFrame({
    'n_obs'     : df_enriched[new_vars].notna().sum(),
    'pct_filled': (df_enriched[new_vars].notna().sum() / len(df_enriched) * 100).round(1),
    'first_obs' : df_enriched[new_vars].apply(lambda c: c.first_valid_index()),
    'last_obs'  : df_enriched[new_vars].apply(lambda c: c.last_valid_index()),
}).sort_values('pct_filled', ascending=False)

print(audit.to_string())

sparse = audit[audit['pct_filled'] < 80]
if not sparse.empty:
    print("\nVariables <80% coverage (based on importance I'll consider forward-filling or exclusion):")
    print(sparse[['pct_filled', 'first_obs']].to_string())
else:
    print('\nAll new variables ≥80% coverage.')

             n_obs  pct_filled  first_obs   last_obs
lgold          385       100.0 1990-01-01 2022-01-01
lip            385       100.0 1990-01-01 2022-01-01
tb3ms          385       100.0 1990-01-01 2022-01-01
gs10           385       100.0 1990-01-01 2022-01-01
tedrate        385       100.0 1990-01-01 2022-01-01
bdi            385       100.0 1990-01-01 2022-01-01
indpro         385       100.0 1990-01-01 2022-01-01
lbdi           385       100.0 1990-01-01 2022-01-01
term_spread    385       100.0 1990-01-01 2022-01-01
vix            384        99.7 1990-03-01 2022-01-01
dvix           383        99.5 1990-04-01 2022-01-01
lreer          336        87.3 1994-03-01 2022-01-01
reer           336        87.3 1994-03-01 2022-01-01
lepu           300        77.9 1997-03-01 2022-01-01
epu            300        77.9 1997-03-01 2022-01-01
gscpi          287        74.5 1998-04-01 2022-01-01
lbrent         174        45.2 2007-09-01 2022-01-01

Variables <80% coverage (based on importance 

### B6. Variable role dictionary

In [45]:
VAR_ROLES = {
    'outcome'           : ['lwti'],
    'treatment'         : ['lpri'],
    'instrument'        : ['d2pri', 'd2pri_jp'],
    'controls_baseline' : ['llwip', 'llgop', 'l2lwip', 'l2lgop', 'dlpri'],
    'controls_alliance' : [c for c in df_enriched.columns if c.startswith('dlpri_')],
    'controls_macro'    : [
        c for c in ['vix', 'dvix', 'gs10', 'tb3ms', 'term_spread', 'tedrate',
                    'lreer', 'lgold', 'lbrent', 'lbdi', 'lip', 'gscpi', 'lepu']
        if c in df_enriched.columns
    ],
}
VAR_ROLES['controls_all_ml'] = (
    VAR_ROLES['controls_baseline']
    + VAR_ROLES['controls_alliance']
    + VAR_ROLES['controls_macro']
)

print('Variable inventory:')
for role, cols in VAR_ROLES.items():
    print(f'  {role:25s}: {cols}')
print(f'\nTotal ML control variables: {len(VAR_ROLES["controls_all_ml"])}')

Variable inventory:
  outcome                  : ['lwti']
  treatment                : ['lpri']
  instrument               : ['d2pri', 'd2pri_jp']
  controls_baseline        : ['llwip', 'llgop', 'l2lwip', 'l2lgop', 'dlpri']
  controls_alliance        : ['dlpri_jp', 'dlpri_aus', 'dlpri_cds', 'dlpri_fra', 'dlpri_ger', 'dlpri_india', 'dlpri_indo', 'dlpri_pak', 'dlpri_rus', 'dlpri_vn', 'dlpri_uk']
  controls_macro           : ['vix', 'dvix', 'gs10', 'tb3ms', 'term_spread', 'tedrate', 'lreer', 'lgold', 'lbrent', 'lbdi', 'lip', 'gscpi', 'lepu']
  controls_all_ml          : ['llwip', 'llgop', 'l2lwip', 'l2lgop', 'dlpri', 'dlpri_jp', 'dlpri_aus', 'dlpri_cds', 'dlpri_fra', 'dlpri_ger', 'dlpri_india', 'dlpri_indo', 'dlpri_pak', 'dlpri_rus', 'dlpri_vn', 'dlpri_uk', 'vix', 'dvix', 'gs10', 'tb3ms', 'term_spread', 'tedrate', 'lreer', 'lgold', 'lbrent', 'lbdi', 'lip', 'gscpi', 'lepu']

Total ML control variables: 29


### B7. Export

In [46]:
OUT_CSV   = f'{PROC}/feature_matrix.csv'
OUT_ROLES = f'{PROC}/var_roles.json'

df_enriched.to_csv(OUT_CSV)
with open(OUT_ROLES, 'w') as fh:
    json.dump(VAR_ROLES, fh, indent=2)

print(f'Feature matrix saved : {OUT_CSV}  ({df_enriched.shape[0]} × {df_enriched.shape[1]})')
print(f'Variable roles saved : {OUT_ROLES}')


Feature matrix saved : ../data/02_features/feature_matrix.csv  (385 × 65)
Variable roles saved : ../data/02_features/var_roles.json


---
## Descriptive stats

In [47]:
macro_vars = VAR_ROLES.get('controls_macro', [])
if macro_vars:
    df_enriched[macro_vars].describe().round(3)

In [48]:
# Correlation with log WTI — exploratory only, not causal
if macro_vars:
    corr = (
        df_enriched[macro_vars + ['lwti']]
        .corr()['lwti']
        .drop('lwti')
        .sort_values()
    )
    print('Correlation with log WTI (exploratory):')
    print(corr.round(3).to_string())

Correlation with log WTI (exploratory):
lreer         -0.549
gs10          -0.499
tb3ms         -0.499
vix           -0.129
tedrate       -0.099
dvix          -0.044
gscpi          0.085
lepu           0.085
term_spread    0.087
lbdi           0.550
lip            0.559
lgold          0.651
lbrent         0.903
